# Classification ECG — Comparaison de 4 modèles
**Modèles :** CNN · BiLSTM · CNN+BiLSTM · ResNet1D

Ce notebook entraîne les 4 modèles et compare automatiquement leurs performances.

In [ ]:
!pip install -q wfdb seaborn scikit-learn

In [ ]:
import os, ast, random, warnings
warnings.filterwarnings('ignore')

import wfdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from scipy.signal import butter, filtfilt

from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv1D, BatchNormalization, MaxPooling1D, Bidirectional, LSTM,
    GlobalAveragePooling1D, Dense, Dropout, Activation, Add
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

SEED = 42
np.random.seed(SEED); random.seed(SEED); tf.random.set_seed(SEED)
print('TensorFlow:', tf.__version__)
print('GPU disponible:', tf.config.list_physical_devices('GPU'))

## 1. Chargement des données

In [ ]:
# Détection automatique du chemin
for dirname, _, filenames in os.walk('/kaggle/input'):
    if 'ptbxl_database.csv' in filenames:
        DATA_PATH = dirname
        break
print('Dataset trouvé :', DATA_PATH)

df = pd.read_csv(os.path.join(DATA_PATH, 'ptbxl_database.csv'))
scp_statements = pd.read_csv(os.path.join(DATA_PATH, 'scp_statements.csv'), index_col=0)
print('Shape :', df.shape)

## 2. Extraction des labels

In [ ]:
def get_superclass(scp_dict_str):
    scp_dict = ast.literal_eval(scp_dict_str)
    best, best_p = None, 0
    for code, p in scp_dict.items():
        if code in scp_statements.index:
            sc = scp_statements.loc[code, 'diagnostic_class']
            if pd.notna(sc) and p >= best_p:
                best_p, best = p, sc
    return best

df['label'] = df['scp_codes'].apply(get_superclass)
df = df[df['label'].notna()].copy()
print('Distribution :')
print(df['label'].value_counts())

## 3. Chargement et prétraitement des signaux

In [ ]:
def bandpass_filter(signal, lowcut=0.5, highcut=40.0, fs=100, order=4):
    nyq = 0.5 * fs
    b, a = butter(order, [lowcut/nyq, highcut/nyq], btype='band')
    return filtfilt(b, a, signal, axis=0)

def load_signal(filename, data_path, target_len=1000, fs=100):
    record = wfdb.rdsamp(os.path.join(data_path, filename))
    signal = record[0]
    signal = bandpass_filter(signal, fs=fs)
    signal = (signal - signal.mean(axis=0)) / (signal.std(axis=0) + 1e-8)
    if signal.shape[0] >= target_len:
        signal = signal[:target_len]
    else:
        pad = np.zeros((target_len - signal.shape[0], signal.shape[1]))
        signal = np.vstack([signal, pad])
    return signal.astype(np.float32)

print('Chargement des signaux (100 Hz)...')
X, valid_idx = [], []
for idx, row in tqdm(df.iterrows(), total=len(df)):
    try:
        sig = load_signal(row['filename_lr'], DATA_PATH, target_len=1000, fs=100)
        X.append(sig)
        valid_idx.append(idx)
    except:
        pass

X = np.array(X, dtype=np.float32)
df_valid = df.loc[valid_idx].reset_index(drop=True)
print(f'Shape X: {X.shape}')  # (N, 1000, 12)

In [ ]:
import wfdb
import matplotlib.pyplot as plt

classes = df_valid['label'].unique()

plt.figure(figsize=(18,10))

for i, cls in enumerate(classes):

    idx = df_valid[df_valid['label']==cls].index[0]

    signal = X[idx]

    plt.subplot(len(classes),1,i+1)
    plt.plot(signal[:,0])  # dérivation I
    plt.title(f"Classe : {cls}")
    plt.tight_layout()

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

sns.countplot(
    x=df_valid['label'],
    order=df_valid['label'].value_counts().index
)

plt.title("Distribution des 5 classes ECG")
plt.xticks(rotation=45)
plt.show()

In [ ]:
idx = 0

raw_record = wfdb.rdsamp(
    os.path.join(DATA_PATH, df_valid.iloc[idx]['filename_lr'])
)

raw_signal = raw_record[0][:1000,0]

filtered_signal = bandpass_filter(
    raw_record[0][:1000],
    fs=100
)[:,0]

plt.figure(figsize=(15,5))

plt.subplot(1,2,1)
plt.plot(raw_signal)
plt.title("Signal ECG brut")

plt.subplot(1,2,2)
plt.plot(filtered_signal)
plt.title("Signal ECG filtré (0.5-40 Hz)")

plt.tight_layout()
plt.show()

## 4. Encodage et split train/val/test

In [ ]:
le = LabelEncoder()
y = le.fit_transform(df_valid['label'])
num_classes = len(le.classes_)
print('Classes :', le.classes_)

test_mask  = df_valid['strat_fold'].isin([9, 10])
val_mask   = df_valid['strat_fold'] == 8
train_mask = ~test_mask & ~val_mask

X_train, y_train = X[train_mask], y[train_mask]
X_val,   y_val   = X[val_mask],   y[val_mask]
X_test,  y_test  = X[test_mask],  y[test_mask]

print(f'Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}')

y_train_oh = tf.keras.utils.to_categorical(y_train, num_classes)
y_val_oh   = tf.keras.utils.to_categorical(y_val,   num_classes)
y_test_oh  = tf.keras.utils.to_categorical(y_test,  num_classes)

cw = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = dict(enumerate(cw))

In [ ]:
import matplotlib.pyplot as plt

lead_names = [
    "I", "II", "III", "aVR", "aVL", "aVF",
    "V1", "V2", "V3", "V4", "V5", "V6"
]

# choisir un ECG
idx = 0
signal = X_test[idx]

plt.figure(figsize=(15, 12))

for i in range(12):
    plt.subplot(12, 1, i + 1)
    plt.plot(signal[:, i])
    plt.ylabel(lead_names[i])
    plt.grid(True, alpha=0.3)

plt.suptitle("Visualisation des 12 dérivations ECG", fontsize=16)
plt.xlabel("Temps")
plt.tight_layout()
plt.show()

## 5. Définition des 4 architectures

In [ ]:
INPUT_SHAPE = (1000, 12)

# ── Modèle 1 : CNN ──────────────────────────────────────────────────
def build_cnn(input_shape, num_classes):
    inp = Input(shape=input_shape)
    x = Conv1D(64, 7, padding='same', activation='relu')(inp)
    x = BatchNormalization()(x)
    x = MaxPooling1D(2)(x)
    x = Conv1D(128, 5, padding='same', activation='relu')(x)
    x = BatchNormalization()(x)
    x = MaxPooling1D(2)(x)
    x = Conv1D(256, 3, padding='same', activation='relu')(x)
    x = BatchNormalization()(x)
    x = GlobalAveragePooling1D()(x)
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.3)(x)
    out = Dense(num_classes, activation='softmax')(x)
    return Model(inp, out, name='CNN')

# ── Modèle 2 : BiLSTM ───────────────────────────────────────────────
def build_bilstm(input_shape, num_classes):
    inp = Input(shape=input_shape)
    x = Bidirectional(LSTM(64, return_sequences=True))(inp)
    x = Bidirectional(LSTM(64))(x)
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.3)(x)
    out = Dense(num_classes, activation='softmax')(x)
    return Model(inp, out, name='BiLSTM')

# ── Modèle 3 : CNN + BiLSTM ─────────────────────────────────────────
def build_cnn_bilstm(input_shape, num_classes):
    inp = Input(shape=input_shape)
    x = Conv1D(64, 7, padding='same', activation='relu')(inp)
    x = BatchNormalization()(x)
    x = MaxPooling1D(2)(x)
    x = Conv1D(128, 5, padding='same', activation='relu')(x)
    x = BatchNormalization()(x)
    x = MaxPooling1D(2)(x)
    x = Bidirectional(LSTM(64, return_sequences=True))(x)
    x = Bidirectional(LSTM(64))(x)
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.3)(x)
    out = Dense(num_classes, activation='softmax')(x)
    return Model(inp, out, name='CNN_BiLSTM')

# ── Modèle 4 : ResNet1D ─────────────────────────────────────────────
def residual_block(x, filters, kernel_size=7, stride=1, dropout=0.2):
    shortcut = x
    x = Conv1D(filters, kernel_size, strides=stride, padding='same', use_bias=False)(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = Dropout(dropout)(x)
    x = Conv1D(filters, kernel_size, padding='same', use_bias=False)(x)
    x = BatchNormalization()(x)
    if stride != 1 or shortcut.shape[-1] != filters:
        shortcut = Conv1D(filters, 1, strides=stride, padding='same', use_bias=False)(shortcut)
        shortcut = BatchNormalization()(shortcut)
    x = Add()([x, shortcut])
    return Activation('relu')(x)

def build_resnet1d(input_shape, num_classes, dropout=0.3):
    inp = Input(shape=input_shape)
    x = Conv1D(64, 15, strides=2, padding='same', use_bias=False)(inp)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = MaxPooling1D(3, strides=2, padding='same')(x)
    x = residual_block(x, 64,  kernel_size=7, dropout=dropout)
    x = residual_block(x, 64,  kernel_size=7, dropout=dropout)
    x = residual_block(x, 128, kernel_size=7, stride=2, dropout=dropout)
    x = residual_block(x, 128, kernel_size=7, dropout=dropout)
    x = residual_block(x, 256, kernel_size=5, stride=2, dropout=dropout)
    x = residual_block(x, 256, kernel_size=5, dropout=dropout)
    x = GlobalAveragePooling1D()(x)
    x = Dense(256, activation='relu')(x)
    x = Dropout(dropout)(x)
    out = Dense(num_classes, activation='softmax')(x)
    return Model(inp, out, name='ResNet1D')

print('4 architectures définies.')

## 6. Fonction d'entraînement générique

In [ ]:
def train_model(model, name, epochs=30, batch_size=32):
    print(f'\n{'='*50}')
    print(f'  Entraînement : {name}')
    print(f'{'='*50}')
    
    model.compile(
        optimizer=Adam(1e-3),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    callbacks = [
        EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True, verbose=0),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, verbose=0),
        ModelCheckpoint(f'best_{name}.keras', monitor='val_accuracy', save_best_only=True, verbose=0)
    ]
    
    history = model.fit(
        X_train, y_train_oh,
        validation_data=(X_val, y_val_oh),
        epochs=epochs,
        batch_size=batch_size,
        class_weight=class_weight_dict,
        callbacks=callbacks,
        verbose=1
    )
    
    # Évaluation sur le test set
    y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
    acc = accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred, average='macro')
    
    print(f'\n✅ {name} — Test Accuracy: {acc*100:.2f}% | F1 macro: {f1*100:.2f}%')
    return history, acc, f1, y_pred

print('Fonction prête.')

## 7. Entraînement des 4 modèles

In [ ]:
results = {}
histories = {}
predictions = {}

# ── CNN ──
model_cnn = build_cnn(INPUT_SHAPE, num_classes)
hist_cnn, acc_cnn, f1_cnn, pred_cnn = train_model(model_cnn, 'CNN')
results['CNN'] = {'accuracy': acc_cnn, 'f1': f1_cnn}
histories['CNN'] = hist_cnn
predictions['CNN'] = pred_cnn

# Définir les variables pour la cellule finale
test_acc_cnn = acc_cnn

In [ ]:
# ── BiLSTM ──
model_bilstm = build_bilstm(INPUT_SHAPE, num_classes)
hist_bilstm, acc_bilstm, f1_bilstm, pred_bilstm = train_model(model_bilstm, 'BiLSTM')
results['BiLSTM'] = {'accuracy': acc_bilstm, 'f1': f1_bilstm}
histories['BiLSTM'] = hist_bilstm
predictions['BiLSTM'] = pred_bilstm

test_acc_bilstm = acc_bilstm

In [ ]:
# ── CNN + BiLSTM ──
model_cnn_bilstm = build_cnn_bilstm(INPUT_SHAPE, num_classes)
hist_cnn_bilstm, acc_cnn_bilstm, f1_cnn_bilstm, pred_cnn_bilstm = train_model(model_cnn_bilstm, 'CNN_BiLSTM')
results['CNN + BiLSTM'] = {'accuracy': acc_cnn_bilstm, 'f1': f1_cnn_bilstm}
histories['CNN + BiLSTM'] = hist_cnn_bilstm
predictions['CNN + BiLSTM'] = pred_cnn_bilstm

test_acc_cnn_bilstm = acc_cnn_bilstm

In [ ]:
# ── ResNet1D ──
model_resnet = build_resnet1d(INPUT_SHAPE, num_classes)
hist_resnet, acc_resnet, f1_resnet, pred_resnet = train_model(model_resnet, 'ResNet1D')
results['ResNet1D'] = {'accuracy': acc_resnet, 'f1': f1_resnet}
histories['ResNet1D'] = hist_resnet
predictions['ResNet1D'] = pred_resnet

test_acc_resnet = acc_resnet

## 8. Tableau comparatif et graphiques

In [ ]:
# ── Tableau comparatif ──
comparison_df = pd.DataFrame([
    {'Modèle': name, 'Test Accuracy (%)': v['accuracy']*100, 'F1 Macro (%)': v['f1']*100}
    for name, v in results.items()
]).sort_values('Test Accuracy (%)', ascending=False).reset_index(drop=True)

print('\n===== COMPARAISON FINALE =====')
display(comparison_df)

# ── Graphique Accuracy & F1 ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63']

modeles = list(results.keys())
accs = [results[m]['accuracy']*100 for m in modeles]
f1s  = [results[m]['f1']*100 for m in modeles]

bars1 = axes[0].bar(modeles, accs, color=colors, edgecolor='white', width=0.6)
axes[0].set_title('Test Accuracy (%)', fontsize=13, fontweight='bold')
axes[0].set_ylim(0, 100)
axes[0].grid(axis='y', alpha=0.4)
for bar, val in zip(bars1, accs):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{val:.1f}%', ha='center', fontsize=11, fontweight='bold')

bars2 = axes[1].bar(modeles, f1s, color=colors, edgecolor='white', width=0.6)
axes[1].set_title('F1 Macro (%)', fontsize=13, fontweight='bold')
axes[1].set_ylim(0, 100)
axes[1].grid(axis='y', alpha=0.4)
for bar, val in zip(bars2, f1s):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{val:.1f}%', ha='center', fontsize=11, fontweight='bold')

plt.suptitle('Comparaison des 4 modèles — PTB-XL ECG', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('comparaison_modeles.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Courbes d'apprentissage

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
colors_map = {'CNN': '#2196F3', 'BiLSTM': '#4CAF50', 'CNN + BiLSTM': '#FF9800', 'ResNet1D': '#E91E63'}

for col, (name, hist) in enumerate(histories.items()):
    c = colors_map[name]
    # Accuracy
    axes[0, col].plot(hist.history['accuracy'], color=c, label='Train')
    axes[0, col].plot(hist.history['val_accuracy'], color=c, linestyle='--', label='Val')
    axes[0, col].set_title(f'{name} — Accuracy', fontweight='bold')
    axes[0, col].legend(fontsize=8)
    axes[0, col].grid(alpha=0.3)
    # Loss
    axes[1, col].plot(hist.history['loss'], color=c, label='Train')
    axes[1, col].plot(hist.history['val_loss'], color=c, linestyle='--', label='Val')
    axes[1, col].set_title(f'{name} — Loss', fontweight='bold')
    axes[1, col].legend(fontsize=8)
    axes[1, col].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('courbes_apprentissage.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Matrices de confusion

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(22, 5))

for ax, (name, y_pred) in zip(axes, predictions.items()):
    cm = confusion_matrix(y_test, y_pred)
    cm_norm = cm.astype(float) / cm.sum(axis=1)[:, None]
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=le.classes_, yticklabels=le.classes_,
                ax=ax, cbar=False)
    ax.set_title(f'{name}\nAcc: {results[name]["accuracy"]*100:.1f}%', fontweight='bold')
    ax.set_xlabel('Prédit')
    ax.set_ylabel('Réel')
    ax.tick_params(axis='x', rotation=45)

plt.suptitle('Matrices de confusion normalisées', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('matrices_confusion.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Rapport détaillé du meilleur modèle

In [ ]:
best_model_name = max(results, key=lambda k: results[k]['accuracy'])
best_pred = predictions[best_model_name]

print(f'🏆 Meilleur modèle : {best_model_name}')
print(f'   Accuracy : {results[best_model_name]["accuracy"]*100:.2f}%')
print(f'   F1 Macro : {results[best_model_name]["f1"]*100:.2f}%')
print()
print('Rapport de classification :')
print(classification_report(y_test, best_pred, target_names=le.classes_))

In [ ]:
from tensorflow.keras.regularizers import l2

def build_cnn_optimized(input_shape, num_classes):

    inp = Input(shape=input_shape)

    x = Conv1D(
        32, 7,
        padding='same',
        activation='relu',
        kernel_regularizer=l2(0.0005)
    )(inp)

    x = BatchNormalization()(x)
    x = MaxPooling1D(2)(x)
    x = Dropout(0.25)(x)

    x = Conv1D(
        64, 5,
        padding='same',
        activation='relu',
        kernel_regularizer=l2(0.0005)
    )(x)

    x = BatchNormalization()(x)
    x = MaxPooling1D(2)(x)
    x = Dropout(0.30)(x)

    x = Conv1D(
        128, 3,
        padding='same',
        activation='relu',
        kernel_regularizer=l2(0.0005)
    )(x)

    x = BatchNormalization()(x)

    x = GlobalAveragePooling1D()(x)

    x = Dense(
        128,
        activation='relu',
        kernel_regularizer=l2(0.0005)
    )(x)

    x = Dropout(0.40)(x)

    out = Dense(num_classes, activation='softmax')(x)

    return Model(inp, out, name="CNN_Optimized")

In [ ]:
model_cnn_opt = build_cnn_optimized(
    INPUT_SHAPE,
    num_classes
)

history_cnn_opt, acc_cnn_opt, f1_cnn_opt, y_pred_cnn_opt = train_model(
    model_cnn_opt,
    "CNN_Optimized",
    epochs=40,
    batch_size=32
)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

classes = np.unique(y_test)

# 1. Courbes Accuracy / Loss
plt.figure(figsize=(14,5))

plt.subplot(1,2,1)
plt.plot(history_cnn_opt.history['accuracy'], label='Train Accuracy')
plt.plot(history_cnn_opt.history['val_accuracy'], label='Validation Accuracy')
plt.title("Courbe Accuracy - CNN Optimized")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()

plt.subplot(1,2,2)
plt.plot(history_cnn_opt.history['loss'], label='Train Loss')
plt.plot(history_cnn_opt.history['val_loss'], label='Validation Loss')
plt.title("Courbe Loss - CNN Optimized")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()

plt.tight_layout()
plt.show()

# 2. Rapport de classification
print("Rapport de classification - CNN Optimized")
print(classification_report(
    y_test,
    y_pred_cnn_opt
))

# 3. Matrice de confusion en nombres
cm = confusion_matrix(y_test, y_pred_cnn_opt)

plt.figure(figsize=(8,6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=classes,
    yticklabels=classes
)

plt.title("Matrice de confusion - CNN Optimized")
plt.xlabel("Classe prédite")
plt.ylabel("Classe réelle")
plt.show()

# 4. Matrice de confusion en pourcentage
cm_percent = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis] * 100

plt.figure(figsize=(8,6))
sns.heatmap(
    cm_percent,
    annot=True,
    fmt=".1f",
    cmap="Blues",
    xticklabels=classes,
    yticklabels=classes
)

plt.title("Matrice de confusion (%) - CNN Optimized")
plt.xlabel("Classe prédite")
plt.ylabel("Classe réelle")
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

comparison_df = pd.DataFrame({
    "Model": ["CNN", "CNN Optimized", "BiLSTM", "CNN+BiLSTM", "ResNet1D"],
    "Accuracy": [
        acc_cnn,
        acc_cnn_opt,
        acc_bilstm,
        acc_cnn_bilstm,
        acc_resnet
    ]
})

plt.figure(figsize=(10,5))
plt.bar(comparison_df["Model"], comparison_df["Accuracy"])

plt.title("Comparaison Accuracy des modèles")
plt.ylabel("Accuracy")
plt.xlabel("Modèle")

for i, v in enumerate(comparison_df["Accuracy"]):
    plt.text(i, v+0.005, f"{v:.3f}", ha='center')

plt.show()

In [ ]:
comparison_df = pd.DataFrame({
    "Model": ["CNN", "CNN Optimized", "BiLSTM", "CNN+BiLSTM", "ResNet1D"],
    "Accuracy": [
        acc_cnn,
        acc_cnn_opt,
        acc_bilstm,
        acc_cnn_bilstm,
        acc_resnet
    ],
    "F1-score": [
        f1_cnn,
        f1_cnn_opt,
        f1_bilstm,
        f1_cnn_bilstm,
        f1_resnet
    ]
})

comparison_df.set_index("Model")[["Accuracy","F1-score"]].plot(
    kind="bar",
    figsize=(10,5)
)

plt.title("Comparaison Accuracy et F1-score")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.5)

plt.show()

In [ ]:
comparison_df = pd.DataFrame({
    "Model": ["CNN", "CNN Optimized", "BiLSTM", "CNN+BiLSTM", "ResNet1D"],
    "Accuracy": [
        acc_cnn,
        acc_cnn_opt,
        acc_bilstm,
        acc_cnn_bilstm,
        acc_resnet
    ],
    "F1-score": [
        f1_cnn,
        f1_cnn_opt,
        f1_bilstm,
        f1_cnn_bilstm,
        f1_resnet
    ]
})

comparison_df.sort_values(
    by="Accuracy",
    ascending=False
)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

models_names = ["CNN", "CNN Optimized", "BiLSTM", "CNN + BiLSTM", "ResNet1D"]

accuracies = [
    acc_cnn * 100,
    acc_cnn_opt * 100,
    acc_bilstm * 100,
    acc_cnn_bilstm * 100,
    acc_resnet * 100
]

f1_scores = [
    f1_cnn * 100,
    f1_cnn_opt * 100,
    f1_bilstm * 100,
    f1_cnn_bilstm * 100,
    f1_resnet * 100
]

colors = ["#1E88E5", "#8E24AA", "#43A047", "#FB8C00", "#E91E63"]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

fig.suptitle("Comparaison des modèles — PTB-XL ECG", fontsize=16, fontweight="bold")

axes[0].bar(models_names, accuracies, color=colors)
axes[0].set_title("Test Accuracy (%)", fontsize=13, fontweight="bold")
axes[0].set_ylim(0, 100)
axes[0].grid(axis="y", alpha=0.3)

for i, v in enumerate(accuracies):
    axes[0].text(i, v + 1, f"{v:.1f}%", ha="center", fontweight="bold")

axes[1].bar(models_names, f1_scores, color=colors)
axes[1].set_title("F1 Macro (%)", fontsize=13, fontweight="bold")
axes[1].set_ylim(0, 100)
axes[1].grid(axis="y", alpha=0.3)

for i, v in enumerate(f1_scores):
    axes[1].text(i, v + 1, f"{v:.1f}%", ha="center", fontweight="bold")

plt.tight_layout()
plt.show()

In [ ]:
print("="*80)
print("CNN")
print("="*80)
model_cnn.summary()

print("\n" + "="*80)
print("CNN Optimized")
print("="*80)
model_cnn_opt.summary()

print("\n" + "="*80)
print("BiLSTM")
print("="*80)
model_bilstm.summary()

print("\n" + "="*80)
print("CNN + BiLSTM")
print("="*80)
model_cnn_bilstm.summary()

print("\n" + "="*80)
print("ResNet1D")
print("="*80)
model_resnet.summary()

In [ ]:
model_cnn.save("cnn_model.keras")

import joblib
joblib.dump(le, "label_encoder.pkl")

In [ ]:
import numpy as np

signal = X_test[0]

np.save("ecg_test.npy", signal)

print(signal.shape)

In [ ]:
for i in range(5):
    np.save(f"ecg_{i}.npy", X_test[i])

    print(
        f"ecg_{i}.npy ->",
        le.inverse_transform([y_test[i]])[0]
    )